###Stream Customers Data From Cloud Files to Delta Lake
---------------------------------------------------------------
1.  Read Files from cloud storage using Data Loader
2. transform the data to add the following columns
    a.file_path: Cloud file path
    b. Ingestion date: Current Timestamp
3. Write the transformed data stream to Delta lake Table

###### - https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader/ 

####1. Read using DataStreamReading API

In [0]:
customer_df = (
                spark.readStream
                    .format("CloudFiles")
                    .option("cloudFiles.format", "json")
                    .option("cloudFiles.schemaLocation", "/Volumes/gizmobox/landing/operational_data/customers_autoloader/_schema")
                    .option("cloudFiles.inferColumnTypes", "true")
                    .option("cloudFiles.schemaHints", "date_of_birth DATE, member_since DATE, created_timestamp TIMESTAMP" )
                    .load("/Volumes/gizmobox/landing/operational_data/customers_autoloader/")
)

####2. Transform the data to add the following columns 
---------------------------------------------------------------
1. file_path: Cloud file path 
2. Ingestion date: Current Timestamp


In [0]:
from pyspark.sql import functions as F
customer_trasformed_df = (
                    customer_df
                    .withColumn("file_path", F.col("_metadata.file_path"))
                    .withColumn("ingestion_date", F.current_timestamp())
)

####3. Write the transformed data stream to Delta lake Table

In [0]:
streaming_query = (
            customer_trasformed_df
            .writeStream.format("delta")
            .trigger(once=True)  ## Other options are "availableNow", "processingTime
            .outputMode("append") ## other options are "update" and "complete"
            .option("checkpointLocation", "/Volumes/gizmobox/landing/operational_data/customers_autoloader/_checkpoint_stream")
            .toTable("gizmobox.bronze.customers_stream")
)

In [0]:
streaming_query = (
            customer_trasformed_df
            .writeStream.format("delta")
            .option("checkpointLocation", "/Volumes/gizmobox/landing/operational_data/customers_autoloader/_checkpoint_stream")
            .toTable("gizmobox.bronze.customers_autoloader")
)

In [0]:
streaming_query.stop()

In [0]:
%sql
SELECT * FROM  gizmobox.bronze.customers_autoloader